# Tripadvisor Scraper v2 — API Oficial — Restaurantes en Bogotá
## Proyecto: DoingEconomics

**Versión 2:** Reemplaza Selenium por la **TripAdvisor Content API oficial**.
Sin bloqueos, sin captchas, datos estructurados en JSON.

---

### Flujo del proceso

```
Base raw (41 restaurantes)
        │
        ▼
[Chunk 8]  Búsqueda de URLs vía Serper.dev
        │
        ▼
[Chunk 9]  Selección del mejor link + extracción de location_id
        │
        ▼
[⚠️ CHECKPOINT]  Revisión manual en Excel
        │
        ▼
[Chunk 10] Append inicial a la base raw
        │
        ▼
[Chunk 11] TripAdvisor Content API
           ├─ Extrae location_id de la URL (-d{id}-)
           ├─ Llama a /location/{id}/details
           └─ Obtiene rating + num_reviews en JSON limpio
        │
        ▼
[Chunk 12] Validación y verificación de resultados API
        │
        ▼
[Chunk 14] Verificación del DataFrame final
        │
        ▼
[Chunk 15] Exportación final a Excel
```

### Cambios respecto a v1
| v1 (Selenium) | v2 (API oficial) |
|---|---|
| Bloqueos de Tripadvisor | Sin bloqueos |
| HTML + BeautifulSoup | JSON estructurado |
| Regex para extraer datos | Campos directos del JSON |
| Archivos `.txt` | Archivos `.json` (trazabilidad) |
| ~4–9 seg/restaurante | ~0.5–1 seg/restaurante |

> 📋 **Prerequisito nuevo:** debes tener una `TRIPADVISOR_API_KEY`.
> Regístrate en: https://tripadvisor-content-api.readme.io
> Los primeros **5.000 calls/mes son gratuitos** (requiere tarjeta de crédito para sobrecargos).
> Para 41 restaurantes usarás máximo **41 llamadas**.


## Chunk 1: Instalación e importación de librerías

> ℹ️ **v2:** Ya no se necesita Selenium ni webdriver-manager.
> Las librerías son más ligeras y no requieren instalar ChromeDriver.


In [1]:
# ── Instalación (descomenta si es la primera vez) ────────────────────────────
# !pip install pandas requests python-dotenv openpyxl beautifulsoup4

# ── Librerías estándar ────────────────────────────────────────────────────────
import os
import re
import json
import time
import random
import logging
from pathlib import Path
from datetime import datetime

# ── Datos y archivos ──────────────────────────────────────────────────────────
import pandas as pd
import openpyxl

# ── Web y entorno ─────────────────────────────────────────────────────────────
import requests
from dotenv import load_dotenv

print("✅ Librerías importadas correctamente.")
print("   (v2: sin Selenium ni webdriver-manager)")


✅ Librerías importadas correctamente.
   (v2: sin Selenium ni webdriver-manager)


## Chunk 2: Definición de rutas del proyecto

In [32]:
# ── Raíz del proyecto ─────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect")

# ── Subcarpetas de trabajo ────────────────────────────────────────────────────
RAW_DIR     = PROJECT_ROOT / "Data" / "Raw"
CLEAN_DIR   = PROJECT_ROOT / "Data" / "Clean"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
JSON_DIR    = OUTPUTS_DIR / "json"   # v2: JSON en lugar de TXT
EXCEL_DIR   = OUTPUTS_DIR / "created_data"
LOGS_DIR    = OUTPUTS_DIR / "logs"

# ── Crear carpetas si no existen ──────────────────────────────────────────────
for folder in [RAW_DIR, JSON_DIR, EXCEL_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Rutas del proyecto configuradas:")
for nombre_ruta, ruta in [
    ("RAW_DIR   ", RAW_DIR),
    ("JSON_DIR  ", JSON_DIR),
    ("EXCEL_DIR ", EXCEL_DIR),
    ("LOGS_DIR  ", LOGS_DIR),
]:
    existe = "✅" if ruta.exists() else "⚠️  (creada)"
    print(f"   {nombre_ruta}: {ruta}  {existe}")


✅ Rutas del proyecto configuradas:
   RAW_DIR   : C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect\Data\Raw  ✅
   JSON_DIR  : C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect\outputs\json  ✅
   EXCEL_DIR : C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect\outputs\created_data  ✅
   LOGS_DIR  : C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect\outputs\logs  ✅


## Chunk 3: Configuración de `.env` y carga de API keys

El archivo `.env` debe tener **dos claves** en la v2:

```
SERPER_API_KEY=tu_api_key_de_serper
TRIPADVISOR_API_KEY=tu_api_key_de_tripadvisor
```

**Cómo obtener la TRIPADVISOR_API_KEY:**
1. Ve a https://tripadvisor-content-api.readme.io
2. Inicia sesión con tu cuenta de Tripadvisor (o crea una)
3. Añade una tarjeta de crédito (para sobrecargos si superas 5.000 calls/mes)
4. Copia tu API key desde el dashboard
5. Pégala en el archivo `.env`

> 💡 Con 41 restaurantes usarás máximo **41 llamadas** — muy por debajo del límite gratuito.


In [6]:
# ── Cargar variables de entorno desde .env ────────────────────────────────────
ENV_PATH = PROJECT_ROOT / "Scripts" / ".env"
load_dotenv(dotenv_path=ENV_PATH)

SERPER_API_KEY       = os.getenv("SERPER_API_KEY")
TRIPADVISOR_API_KEY  = os.getenv("TRIPADVISOR_API_KEY")

# ── Validar Serper ────────────────────────────────────────────────────────────
if not SERPER_API_KEY:
    raise EnvironmentError(
        f"\n❌ No se encontró SERPER_API_KEY en: {ENV_PATH}"
    )

# ── Validar Tripadvisor ───────────────────────────────────────────────────────
if not TRIPADVISOR_API_KEY:
    raise EnvironmentError(
        f"\n❌ No se encontró TRIPADVISOR_API_KEY en: {ENV_PATH}\n"
        f"   Regístrate en: https://tripadvisor-content-api.readme.io\n"
        f"   Y añade: TRIPADVISOR_API_KEY=tu_api_key_aqui"
    )

print("✅ API keys cargadas correctamente.")
print(f"   SERPER_API_KEY      : ...{SERPER_API_KEY[-4:]}")
print(f"   TRIPADVISOR_API_KEY : ...{TRIPADVISOR_API_KEY[-4:]}")


✅ API keys cargadas correctamente.
   SERPER_API_KEY      : ...2750
   TRIPADVISOR_API_KEY : ...06BC


## Chunk 4: Carga de la base raw desde `Data/Raw`

> ✏️ **Acción requerida:** Cambia `nombre_de_tu_base.xlsx` por el nombre real de tu archivo.


In [8]:
# ── CAMBIA ESTE NOMBRE por el de tu archivo raw ───────────────────────────────
raw_file = RAW_DIR / "base_raw.xlsx"

if not raw_file.exists():
    raise FileNotFoundError(
        f"\n❌ Archivo no encontrado:\n   {raw_file}\n"
        f"   Verifica el nombre y que esté en Data/Raw."
    )

df_raw = pd.read_excel(raw_file)

print(f"✅ Base raw cargada exitosamente.")
print(f"   Archivo : {raw_file.name}")
print(f"   Filas   : {len(df_raw)}")
print(f"   Columnas: {len(df_raw.columns)}")
df_raw.head(3)


✅ Base raw cargada exitosamente.
   Archivo : base_raw.xlsx
   Filas   : 41
   Columnas: 16


,id,place_id,nombre,direccion,lat,lon,rating,num_reviews,google_maps_url,tipos,query_origen,barrio,cocina,no_visitar,sample,encuesta
0,1,ChIJUcItNvWaP44RRi9_rm9immw,Vapiano Colombia Restaurante Italiano,"Cra. 12a #83 - 11, Vapiano, Bogotá, Colombia",4.667606,-74.054134,4.7,10053,https://maps.google.com/?cid=78256755343181207...,"italian_restaurant, restaurant, point_of_inter...",restaurante italiano Bogotá,El Retiro,Italiana,NaN,1,1
1,2,ChIJv5caPvybP44RcRWTKtso-Fo,Storia D'Amore zona T,"Kr 13 #82-36, Bogotá, Colombia",4.667219,-74.054657,4.7,7987,https://maps.google.com/?cid=65550341794149881...,"italian_restaurant, restaurant, point_of_inter...",restaurante italiano Bogotá,El Retiro,Italiana,NaN,1,1
2,5,ChIJtfDZiJqZP44R-G_-QmrdkM0,Restaurante español Gaudí,"Cra. 4a #27-54, Santa Fé, Bogotá, Cundinamarca...",4.614387,-74.066147,4.6,4779,https://maps.google.com/?cid=14812582622881804...,"spanish_restaurant, tapas_restaurant, mediterr...",restaurante español Bogotá Colombia,La Macarena,Española,NaN,1,1


## Chunk 5: Configuración de columnas de la base raw

In [9]:
# ── Variables de referencia ───────────────────────────────────────────────────
ID_COL        = "id"
NAME_COL      = "nombre"
ADDRESS_COL   = "direccion"
CITY_VALUE    = "Bogotá"
COUNTRY_VALUE = "Colombia"

# ── Columnas originales (NO se modificarán) ───────────────────────────────────
ORIGINAL_COLS = [
    "id", "place_id", "nombre", "direccion", "lat", "lon",
    "rating", "num_reviews", "google_maps_url", "tipos",
    "query_origen", "barrio", "cocina", "no_visitar", "sample", "encuesta"
]

# ── Columnas nuevas que se añadirán al final ──────────────────────────────────
# NOTA: tripadvisor_txt_file ahora apunta a un archivo .json (no .txt)
NEW_COLS = [
    "tripadvisor",
    "tripadvisor_url",
    "tripadvisor_location_id",   # v2: nuevo campo — ID de Tripadvisor extraído de la URL
    "tripadvisor_rating",
    "tripadvisor_n_reviews",
    "tripadvisor_ranking",       # v2: nuevo campo — ranking del restaurante en Bogotá
    "tripadvisor_status",
    "tripadvisor_error",
    "tripadvisor_txt_file"       # apunta al .json guardado (mantiene compatibilidad)
]

print("✅ Variables de columnas configuradas.")
print(f"   Columnas originales : {len(ORIGINAL_COLS)}")
print(f"   Columnas nuevas     : {len(NEW_COLS)}")


✅ Variables de columnas configuradas.
   Columnas originales : 16
   Columnas nuevas     : 9


## Chunk 6: Validación de columnas esperadas

In [10]:
print("🔍 Validando columnas de la base raw...\n")

for col in ORIGINAL_COLS:
    estado = "✅" if col in df_raw.columns else "❌  FALTANTE"
    print(f"   {estado}  {col}")

faltantes = [c for c in ORIGINAL_COLS if c not in df_raw.columns]
if faltantes:
    print(f"\n⚠️  Columnas faltantes: {faltantes}")
else:
    print(f"\n✅ Todas las {len(ORIGINAL_COLS)} columnas originales están presentes.")


🔍 Validando columnas de la base raw...

   ✅  id
   ✅  place_id
   ✅  nombre
   ✅  direccion
   ✅  lat
   ✅  lon
   ✅  rating
   ✅  num_reviews
   ✅  google_maps_url
   ✅  tipos
   ✅  query_origen
   ✅  barrio
   ✅  cocina
   ✅  no_visitar
   ✅  sample
   ✅  encuesta

✅ Todas las 16 columnas originales están presentes.


## Chunk 7: Funciones auxiliares generales

> ℹ️ **v2:** Se eliminaron `configurar_chrome` y `limpiar_html_a_texto`.
> Se añadió `extraer_location_id` para parsear el ID desde la URL de Tripadvisor.


In [11]:
def clean_filename(text: str) -> str:
    """Convierte texto en nombre de archivo válido para Windows (max 80 chars)."""
    text = str(text).lower().strip()
    for src, dst in [
        ("áàäâ","a"),("éèëê","e"),("íìïî","i"),
        ("óòöô","o"),("úùüû","u"),("ñ","n")
    ]:
        for ch in src:
            text = text.replace(ch, dst)
    text = re.sub(r"[^a-z0-9\s]", "_", text)
    text = re.sub(r"[\s_]+", "_", text)
    return text.strip("_")[:80]


def random_delay(min_s: float = 0.3, max_s: float = 1.2):
    """
    Pausa aleatoria entre llamadas a la API.
    La API de Tripadvisor es estable — delays cortos son suficientes.
    """
    time.sleep(round(random.uniform(min_s, max_s), 2))


def log_event(log_list: list, id_val, nombre: str,
              etapa: str, status: str, mensaje: str = ""):
    """Registra un evento en la lista de logs con timestamp."""
    log_list.append({
        "id":            id_val,
        "nombre":        nombre,
        "etapa":         etapa,
        "status":        status,
        "mensaje_error": mensaje,
        "timestamp":     datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })


def extraer_location_id(url: str) -> str | None:
    """
    Extrae el location_id desde una URL de Tripadvisor.

    Las URLs de Tripadvisor tienen el patrón:
      .../Restaurant_Review-g{geo_id}-d{location_id}-Reviews-...
    
    Ejemplo:
      tripadvisor.co/Restaurant_Review-g294074-d4748184-Reviews-Trattoria...
      → location_id = "4748184"

    Retorna el location_id como string, o None si no se encuentra.
    """
    if not url:
        return None
    match = re.search(r"-d(\d+)-", str(url))
    return match.group(1) if match else None


def is_tripadvisor_restaurant_url(url: str) -> bool:
    """Verifica si una URL es una página de restaurante en Tripadvisor."""
    if not url or "tripadvisor" not in url.lower():
        return False
    return "restaurant_review" in url.lower()


# Inicializar lista global de logs
global_logs = []

print("✅ Funciones auxiliares definidas:")
print("   clean_filename(text)")
print("   random_delay(min_s, max_s)")
print("   log_event(log_list, id_val, nombre, etapa, status, mensaje)")
print("   extraer_location_id(url)  ← nueva en v2")
print("   is_tripadvisor_restaurant_url(url)")


✅ Funciones auxiliares definidas:
   clean_filename(text)
   random_delay(min_s, max_s)
   log_event(log_list, id_val, nombre, etapa, status, mensaje)
   extraer_location_id(url)  ← nueva en v2
   is_tripadvisor_restaurant_url(url)


## Chunk 8: Búsqueda de links de Tripadvisor con Serper.dev

Igual que en v1. Para cada restaurante se construyen 1–2 queries y se extraen
links candidatos de Tripadvisor.

> 💡 **Tip v2:** Ahora el location_id se puede validar en este paso: si la URL
> tiene el patrón `-d{id}-`, se puede confirmar que es una página válida de restaurante.


In [12]:
def buscar_links_tripadvisor(row: pd.Series, api_key: str) -> list:
    """
    Busca links de Tripadvisor para un restaurante usando la API de Serper.dev.
    Retorna una lista de candidatos con metadata, incluyendo el location_id extraído.
    """
    nombre    = str(row[NAME_COL])
    id_val    = row[ID_COL]
    dir_raw   = row.get(ADDRESS_COL, "")
    direccion = (
        str(dir_raw).strip()
        if pd.notna(dir_raw) and str(dir_raw).strip() not in ["", "nan"]
        else ""
    )

    queries = [f"{nombre} Tripadvisor {CITY_VALUE} {COUNTRY_VALUE}"]
    if len(direccion) > 5:
        queries.append(f"{nombre} {direccion} Tripadvisor {CITY_VALUE}")

    candidatos = []

    for q_idx, query in enumerate(queries):
        try:
            response = requests.post(
                "https://google.serper.dev/search",
                headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
                json={"q": query, "gl": "co", "hl": "es", "num": 10},
                timeout=15
            )
            if response.status_code != 200:
                print(f"      ⚠️  Error API Serper [{response.status_code}]")
                continue

            for rank, result in enumerate(response.json().get("organic", [])):
                url       = result.get("link", "")
                title     = result.get("title", "")
                snippet   = result.get("snippet", "")
                is_ta     = "tripadvisor" in url.lower()
                is_rest   = is_tripadvisor_restaurant_url(url)
                loc_id    = extraer_location_id(url) if is_rest else None  # ← v2

                candidatos.append({
                    "id":                 id_val,
                    "nombre":             nombre,
                    "direccion":          direccion,
                    "query_usada":        query,
                    "candidate_rank":     rank + 1,
                    "candidate_url":      url,
                    "candidate_title":    title,
                    "candidate_snippet":  snippet,
                    "is_tripadvisor":     is_ta,
                    "is_restaurant_page": is_rest,
                    "location_id":        loc_id,    # ← v2: nuevo campo
                    "selected_candidate": False
                })

            if len(queries) > 1 and q_idx < len(queries) - 1:
                random_delay(1.5, 2.5)

        except requests.exceptions.Timeout:
            print(f"      ❌ Timeout en: {query[:60]}...")
        except Exception as e:
            print(f"      ❌ Excepción: {e}")

    return candidatos


In [13]:
# ── Ejecutar búsqueda para los 41 restaurantes ────────────────────────────────
print(f"🔍 Iniciando búsqueda en Serper.dev — {len(df_raw)} restaurantes")
print(f"   Geolocalización: Colombia (gl=co) | Idioma: Español (hl=es)")
print("-" * 65)

all_candidatos = []

for i, (idx, row) in enumerate(df_raw.iterrows()):
    nombre = str(row[NAME_COL])
    id_val = row[ID_COL]

    print(f"  [{i+1:>2}/{len(df_raw)}] {nombre}")
    candidatos = buscar_links_tripadvisor(row, SERPER_API_KEY)
    all_candidatos.extend(candidatos)

    n_ta   = sum(1 for c in candidatos if c["is_tripadvisor"])
    n_rest = sum(1 for c in candidatos if c["is_restaurant_page"])
    n_id   = sum(1 for c in candidatos if c.get("location_id"))
    print(f"         → TA: {n_ta} | Pág. restaurante: {n_rest} | Con location_id: {n_id}")

    log_event(global_logs, id_val, nombre, "busqueda_serper",
              "ok" if n_ta > 0 else "sin_resultados_ta",
              f"{n_ta} links Tripadvisor")

    if i < len(df_raw) - 1:
        random_delay(2.0, 4.5)

df_candidatos = pd.DataFrame(all_candidatos)

candidatos_path = EXCEL_DIR / "links_candidatos_tripadvisor.xlsx"
df_candidatos.to_excel(candidatos_path, index=False)

print("\n" + "-" * 65)
print("✅ Búsqueda completada.")
print(f"   Total candidatos       : {len(df_candidatos)}")
print(f"   Links Tripadvisor      : {df_candidatos['is_tripadvisor'].sum()}")
print(f"   Con location_id válido : {df_candidatos['location_id'].notna().sum()}")
print(f"   Tabla guardada en      : {candidatos_path}")


🔍 Iniciando búsqueda en Serper.dev — 41 restaurantes
   Geolocalización: Colombia (gl=co) | Idioma: Español (hl=es)
-----------------------------------------------------------------
  [ 1/41] Vapiano Colombia Restaurante Italiano
         → TA: 13 | Pág. restaurante: 8 | Con location_id: 8
  [ 2/41] Storia D'Amore zona T
         → TA: 12 | Pág. restaurante: 7 | Con location_id: 7
  [ 3/41] Restaurante español Gaudí
         → TA: 11 | Pág. restaurante: 6 | Con location_id: 6
  [ 4/41] La Fabbrica
         → TA: 11 | Pág. restaurante: 4 | Con location_id: 4
  [ 5/41] Griego Rooftop | Crossover Rooftop & Nightclub in Bogotá
         → TA: 9 | Pág. restaurante: 0 | Con location_id: 0
  [ 6/41] Salonika
         → TA: 11 | Pág. restaurante: 6 | Con location_id: 6
  [ 7/41] Primi
         → TA: 13 | Pág. restaurante: 5 | Con location_id: 5
  [ 8/41] Santorini Rooftop
         → TA: 10 | Pág. restaurante: 6 | Con location_id: 6
  [ 9/41] Huerta Coctelería Artesanal
         → TA: 16 | Pág. 

## Chunk 9: Consolidación y selección del mejor link por restaurante

> ℹ️ **v2:** La selección ahora **prioriza candidatos que ya tienen `location_id`** extraído
> (patrón `-d{id}-` en la URL), ya que esos son los más fáciles de consultar vía API.


In [14]:
def seleccionar_mejor_link(grupo: pd.DataFrame, nombre_rest: str) -> dict | None:
    """
    Selecciona el mejor candidato de Tripadvisor para un restaurante.

    Criterios de puntuación (v2):
    - is_restaurant_page == True  → +10  (patrón Restaurant_Review en URL)
    - location_id no nulo         → +8   (v2: confirma URL parseable por la API)
    - 'bogota' en URL             → +5
    - 'colombia' en URL/título    → +3
    - Palabras del nombre en título → +3 por palabra
    - Penalización por rank alto  → −0.5 × rank
    """
    ta = grupo[grupo["is_tripadvisor"] == True].copy()
    if ta.empty:
        return None

    nombre_l = nombre_rest.lower()
    palabras = [p for p in nombre_l.split() if len(p) > 3]

    def calcular_score(row) -> float:
        score     = 0.0
        url_l     = str(row["candidate_url"]).lower()
        title_l   = str(row["candidate_title"]).lower()
        snippet_l = str(row["candidate_snippet"]).lower()

        if row["is_restaurant_page"]:              score += 10
        if pd.notna(row.get("location_id")):       score += 8   # ← v2
        if "bogota" in url_l or "bogot" in url_l:  score += 5
        if "colombia" in url_l or "colombia" in title_l: score += 3
        for p in palabras:
            if p in title_l:    score += 3
            elif p in snippet_l: score += 1
        score -= row["candidate_rank"] * 0.5
        return score

    ta = ta.copy()
    ta["score"] = ta.apply(calcular_score, axis=1)
    mejor = ta.sort_values("score", ascending=False).iloc[0]

    return {
        "candidate_url":      mejor["candidate_url"],
        "candidate_title":    mejor["candidate_title"],
        "candidate_snippet":  mejor["candidate_snippet"],
        "query_usada":        mejor["query_usada"],
        "score":              mejor["score"],
        "is_restaurant_page": mejor["is_restaurant_page"],
        "location_id":        mejor.get("location_id")   # ← v2
    }


print("🎯 Seleccionando mejor link por restaurante...\n")
print("-" * 65)

seleccionados = []

for idx, row in df_raw.iterrows():
    id_val  = row[ID_COL]
    nombre  = str(row[NAME_COL])
    dir_val = row.get(ADDRESS_COL, "")

    grupo = df_candidatos[df_candidatos["id"] == id_val]
    mejor = seleccionar_mejor_link(grupo, nombre)

    if mejor:
        loc_id = mejor.get("location_id")
        # Si no tiene location_id en el resultado, intentar extraerlo de la URL
        if not loc_id:
            loc_id = extraer_location_id(mejor["candidate_url"])

        status = "found" if mejor["is_restaurant_page"] else "found_uncertain"
        # Sin location_id la API no puede funcionar → marcar como incierto
        if not loc_id:
            status = "found_no_id"

        seleccionados.append({
            "id":               id_val,
            "nombre":           nombre,
            "direccion":        dir_val,
            "tripadvisor_url":  mejor["candidate_url"],
            "location_id":      loc_id,           # ← v2
            "match_status":     status,
            "query_usada":      mejor["query_usada"],
            "score":            round(mejor["score"], 2),
            "is_restaurant_page": mejor["is_restaurant_page"],
            "candidate_title":  mejor["candidate_title"]
        })

        icono = "✅" if status == "found" else "⚠️ "
        id_str = f"location_id={loc_id}" if loc_id else "⚠️  SIN location_id"
        print(f"  {icono} [{status}] {nombre}")
        print(f"     URL : {mejor['candidate_url'][:70]}...")
        print(f"     {id_str} | Score: {mejor['score']:.1f}\n")
    else:
        seleccionados.append({
            "id": id_val, "nombre": nombre, "direccion": dir_val,
            "tripadvisor_url": None, "location_id": None,
            "match_status": "not_found", "query_usada": None,
            "score": 0, "is_restaurant_page": False, "candidate_title": None
        })
        print(f"  ❌ [not_found] {nombre}\n")

df_seleccionados = pd.DataFrame(seleccionados)

seleccionados_path = EXCEL_DIR / "links_seleccionados_tripadvisor.xlsx"
df_seleccionados.to_excel(seleccionados_path, index=False)

print("-" * 65)
print("✅ Selección completada.")
print(f"   found          : {(df_seleccionados['match_status']=='found').sum()}")
print(f"   found_uncertain: {(df_seleccionados['match_status']=='found_uncertain').sum()}")
print(f"   found_no_id    : {(df_seleccionados['match_status']=='found_no_id').sum()}")
print(f"   not_found      : {(df_seleccionados['match_status']=='not_found').sum()}")
print(f"   Con location_id: {df_seleccionados['location_id'].notna().sum()}")


🎯 Seleccionando mejor link por restaurante...

-----------------------------------------------------------------
  ✅ [found] Vapiano Colombia Restaurante Italiano
     URL : https://www.tripadvisor.es/Restaurant_Review-g294074-d12657249-Reviews...
     location_id=12657249 | Score: 31.5

  ✅ [found] Storia D'Amore zona T
     URL : https://www.tripadvisor.co/Restaurant_Review-g294074-d21340928-Reviews...
     location_id=21340928 | Score: 31.5

  ✅ [found] Restaurante español Gaudí
     URL : https://www.tripadvisor.com.mx/Restaurant_Review-g294074-d952753-Revie...
     location_id=952753 | Score: 27.0

  ✅ [found] La Fabbrica
     URL : https://www.tripadvisor.co/Restaurant_Review-g294074-d1890139-Reviews-...
     location_id=1890139 | Score: 25.5

  ⚠️  [found_uncertain] Griego Rooftop | Crossover Rooftop & Nightclub in Bogotá
     URL : https://www.tripadvisor.es/Attraction_Review-g294074-d27502790-Reviews...
     location_id=nan | Score: 17.0

  ✅ [found] Salonika
     URL : https:

## ⚠️ CHECKPOINT — Revisión manual obligatoria

**No ejecutes el Chunk 10 hasta completar esta revisión.**

### Pasos:
1. Abre: `outputs/excel/links_seleccionados_tripadvisor.xlsx`
2. Verifica la columna `tripadvisor_url` → ¿la URL abre el restaurante correcto?
3. Verifica la columna `location_id` → debe tener un número (ej. `4748184`)
4. **`match_status = found_no_id`** → la URL existe pero no tiene location_id:
   - Ábrela en el navegador, copia la URL final (puede haber redirect)
   - Busca el patrón `-d{número}-` en la URL y pega el número en la columna `location_id`
5. **`match_status = not_found`** → puedes buscar manualmente en Tripadvisor:
   - Pega la URL en `tripadvisor_url` y el número de la URL en `location_id`
6. Guarda el Excel y ejecuta **Chunk 9b** para recargar.
7. Luego continúa con el **Chunk 10**.


In [15]:
print("=" * 65)
print("  ⏸️  CHECKPOINT: REVISIÓN MANUAL REQUERIDA")
print("=" * 65)
print(f"""
Archivo a revisar:
  {seleccionados_path}

Columnas clave:
  tripadvisor_url  → ¿URL correcta del restaurante?
  location_id      → ¿Tiene un número? (ej: 4748184)
  match_status     → found / found_uncertain / found_no_id / not_found

Casos especiales:
  found_no_id  → Busca '-d{{número}}-' en la URL y añade el número
                 en la columna 'location_id'
  not_found    → Busca el restaurante en tripadvisor.co/es,
                 copia la URL y extrae el número '-d{{id}}-'

⚠️  Sin location_id, la API no puede obtener los datos.
⚠️  NO ejecutes el Chunk 10 hasta completar esta revisión.
""")
print("=" * 65)


  ⏸️  CHECKPOINT: REVISIÓN MANUAL REQUERIDA

Archivo a revisar:
  C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect\outputs\excel\links_seleccionados_tripadvisor.xlsx

Columnas clave:
  tripadvisor_url  → ¿URL correcta del restaurante?
  location_id      → ¿Tiene un número? (ej: 4748184)
  match_status     → found / found_uncertain / found_no_id / not_found

Casos especiales:
  found_no_id  → Busca '-d{número}-' en la URL y añade el número
                 en la columna 'location_id'
  not_found    → Busca el restaurante en tripadvisor.co/es,
                 copia la URL y extrae el número '-d{id}-'

⚠️  Sin location_id, la API no puede obtener los datos.
⚠️  NO ejecutes el Chunk 10 hasta completar esta revisión.



### Chunk 9b: Recarga después de revisión manual

In [21]:
# Recargar después de editar el Excel
df_seleccionados = pd.read_excel(seleccionados_path)
# Convertir location_id a string limpio (por si Excel lo cambió a float)
df_seleccionados["location_id"] = (
    df_seleccionados["location_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .replace("nan", None)
)

print("✅ Tabla recargada después de revisión manual.")
print(f"   Con URL válida  : {df_seleccionados['tripadvisor_url'].notna().sum()}")
print(f"   Con location_id : {df_seleccionados['location_id'].notna().sum()}")
print(f"   Sin location_id : {df_seleccionados['location_id'].isna().sum()}")
print()
display(df_seleccionados[["id","nombre","location_id","tripadvisor_url","match_status"]].head(15))


✅ Tabla recargada después de revisión manual.
   Con URL válida  : 41
   Con location_id : 35
   Sin location_id : 6



,id,nombre,location_id,tripadvisor_url,match_status
0,1,Vapiano Colombia Restaurante Italiano,12657249,https://www.tripadvisor.es/Restaurant_Review-g...,found
1,2,Storia D'Amore zona T,21340928,https://www.tripadvisor.co/Restaurant_Review-g...,found
2,5,Restaurante español Gaudí,952753,https://www.tripadvisor.com.mx/Restaurant_Revi...,found
3,9,La Fabbrica,1890139,https://www.tripadvisor.co/Restaurant_Review-g...,found
4,15,Griego Rooftop | Crossover Rooftop & Nightclub...,27502790,https://www.tripadvisor.es/Attraction_Review-g...,found_uncertain
5,16,Salonika,12906353,https://www.tripadvisor.co/Restaurant_Review-g...,found
6,17,Primi,6930144,https://www.tripadvisor.co/Restaurant_Review-g...,found
7,19,Santorini Rooftop,25246580,https://www.tripadvisor.co/Restaurant_Review-g...,found
8,21,Huerta Coctelería Artesanal,26627757,https://www.tripadvisor.co/Restaurant_Review-g...,found_uncertain
9,22,La Tapería,2214888,https://www.tripadvisor.co/Restaurant_Review-g...,found


## Chunk 10: Append inicial de presencia en Tripadvisor a la base raw

Igual que en v1. Se añaden las columnas nuevas al DataFrame, manteniendo
todas las columnas originales intactas.


In [22]:
# ── Crear copia de trabajo de la base raw ─────────────────────────────────────
df_work = df_raw.copy()

# ── Añadir columnas nuevas vacías ─────────────────────────────────────────────
for col in NEW_COLS:
    if col not in df_work.columns:
        df_work[col] = None

# ── Poblar columnas desde df_seleccionados ────────────────────────────────────
for _, sel_row in df_seleccionados.iterrows():
    id_val = sel_row["id"]
    url    = sel_row.get("tripadvisor_url")
    loc_id = sel_row.get("location_id")
    status = sel_row.get("match_status", "not_found")

    mascara = df_work[ID_COL] == id_val

    if pd.notna(url) and url:
        df_work.loc[mascara, "tripadvisor"]             = 1
        df_work.loc[mascara, "tripadvisor_url"]         = url
        df_work.loc[mascara, "tripadvisor_location_id"] = loc_id
        df_work.loc[mascara, "tripadvisor_status"]      = status
    else:
        df_work.loc[mascara, "tripadvisor"]             = 0
        df_work.loc[mascara, "tripadvisor_url"]         = None
        df_work.loc[mascara, "tripadvisor_location_id"] = None
        df_work.loc[mascara, "tripadvisor_status"]      = "not_found"

cols_ok = all(c in df_work.columns for c in ORIGINAL_COLS)
print("✅ Append inicial completado.")
print(f"   Con Tripadvisor (1)          : {(df_work['tripadvisor']==1).sum()}")
print(f"   Sin Tripadvisor (0)          : {(df_work['tripadvisor']==0).sum()}")
print(f"   Con location_id              : {df_work['tripadvisor_location_id'].notna().sum()}")
print(f"   Columnas originales intactas : {'✅ Sí' if cols_ok else '❌ Revisar'}")


✅ Append inicial completado.
   Con Tripadvisor (1)          : 41
   Sin Tripadvisor (0)          : 0
   Con location_id              : 35
   Columnas originales intactas : ✅ Sí


## Chunk 11: Consulta a la TripAdvisor Content API ✨

**Este chunk reemplaza completamente a Selenium.**

### Qué hace:
1. Para cada restaurante con `tripadvisor == 1` y `tripadvisor_location_id` válido
2. Extrae el `location_id` de la URL (patrón `-d{id}-`)
3. Llama a `GET /api/v1/location/{location_id}/details`
4. Extrae `rating` y `num_reviews` directamente del JSON
5. Guarda la respuesta JSON completa en `outputs/json/` (trazabilidad académica)

### Campos que devuelve la API:
```json
{
  "rating": "4.5",
  "num_reviews": "334",
  "ranking_data": {
    "ranking_string": "N.º 252 de 5.839 restaurantes en Bogotá",
    "ranking": "252",
    "ranking_denominator": "5839"
  },
  "name": "Trattoria la divina comedia",
  "price_level": "$$-$$$",
  "cuisine": [{"localized_name": "Italiana"}, ...]
}
```

> ⏱️ Con delays de 0.5–1.2 seg, los 41 restaurantes se procesan en **~1 minuto**.
> Comparado con ~5–8 min con Selenium (cuando no bloqueaba).


In [23]:
# ── Función principal de la API ───────────────────────────────────────────────
TA_API_BASE = "https://api.content.tripadvisor.com/api/v1"

def obtener_detalles_tripadvisor(location_id: str, api_key: str) -> dict:
    """
    Llama al endpoint Location Details de la TripAdvisor Content API.

    Endpoint: GET /location/{location_id}/details
    Documentación: https://tripadvisor-content-api.readme.io/reference/getlocationdetails

    Parámetros:
        location_id : ID numérico del lugar en Tripadvisor (extraído de la URL)
        api_key     : TRIPADVISOR_API_KEY del archivo .env

    Retorna:
        Dict con los detalles del restaurante (rating, num_reviews, ranking, etc.)

    Errores:
        404 → location_id no encontrado (URL incorrecta o restaurante eliminado)
        429 → rate limit superado (poco probable con 41 restaurantes)
        401 → API key inválida
    """
    endpoint = f"{TA_API_BASE}/location/{location_id}/details"
    params   = {
        "key":      api_key,
        "language": "es",
        "currency": "COP"
    }

    response = requests.get(endpoint, params=params, timeout=15)
    response.raise_for_status()   # Lanza excepción si status != 200
    return response.json()


def parsear_respuesta_api(data: dict) -> dict:
    """
    Extrae los campos de interés de la respuesta JSON de la API.

    Retorna un dict con:
        rating        : calificación promedio (str, ej. "4.5")
        n_reviews     : número de opiniones (int, ej. 334)
        ranking_str   : string de ranking (ej. "N.º 252 de 5.839 restaurantes en Bogotá")
        nombre_api    : nombre del restaurante según Tripadvisor
        price_level   : nivel de precios (ej. "$$-$$$")
    """
    ranking_data = data.get("ranking_data", {}) or {}

    # num_reviews puede venir como string o int según la versión de la API
    raw_reviews = data.get("num_reviews", None)
    try:
        n_reviews = int(str(raw_reviews).replace(",", "").replace(".", "")) if raw_reviews else None
    except (ValueError, TypeError):
        n_reviews = None

    return {
        "rating":      data.get("rating", None),
        "n_reviews":   n_reviews,
        "ranking_str": ranking_data.get("ranking_string", None),
        "nombre_api":  data.get("name", None),
        "price_level": data.get("price_level", None)
    }


In [24]:
# ── Ejecutar consulta a la API para todos los restaurantes ────────────────────
df_con_ta = df_work[
    (df_work["tripadvisor"] == 1) &
    (df_work["tripadvisor_location_id"].notna())
].copy()

df_sin_id = df_work[
    (df_work["tripadvisor"] == 1) &
    (df_work["tripadvisor_location_id"].isna())
].copy()

print(f"📡 Consultando TripAdvisor Content API")
print(f"   Restaurantes a consultar       : {len(df_con_ta)}")
print(f"   Sin location_id (se omiten)    : {len(df_sin_id)}")
print(f"   API Key                        : ...{TRIPADVISOR_API_KEY[-4:]}")
print("-" * 65)

if len(df_sin_id) > 0:
    print(f"\n  ⚠️  Restaurantes SIN location_id (requieren revisión manual):")
    for _, r in df_sin_id.iterrows():
        print(f"     - {r[NAME_COL]} | URL: {str(r['tripadvisor_url'])[:60]}...")
    print()

for i, (idx, row) in enumerate(df_con_ta.iterrows()):
    id_val   = row[ID_COL]
    nombre   = str(row[NAME_COL])
    loc_id   = str(row["tripadvisor_location_id"])
    url      = row["tripadvisor_url"]

    print(f"  [{i+1:>2}/{len(df_con_ta)}] {nombre}")
    print(f"   location_id: {loc_id}")

    try:
        # ── Llamar a la API ───────────────────────────────────────────────────
        data     = obtener_detalles_tripadvisor(loc_id, TRIPADVISOR_API_KEY)
        parsed   = parsear_respuesta_api(data)

        # ── Actualizar df_work ────────────────────────────────────────────────
        df_work.loc[idx, "tripadvisor_rating"]    = parsed["rating"]
        df_work.loc[idx, "tripadvisor_n_reviews"] = parsed["n_reviews"]
        df_work.loc[idx, "tripadvisor_ranking"]   = parsed["ranking_str"]
        df_work.loc[idx, "tripadvisor_status"]    = "data_extracted"
        df_work.loc[idx, "tripadvisor_error"]     = None

        # ── Guardar JSON completo (trazabilidad académica) ────────────────────
        filename  = f"{id_val}_{clean_filename(nombre)}.json"
        json_path = JSON_DIR / filename
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        df_work.loc[idx, "tripadvisor_txt_file"] = str(json_path)

        log_event(global_logs, id_val, nombre, "tripadvisor_api", "data_extracted",
                  f"rating={parsed['rating']} | n_reviews={parsed['n_reviews']}")

        print(f"   ✅ rating={parsed['rating']} | reviews={parsed['n_reviews']} | {parsed['ranking_str']}")
        print(f"   💾 JSON: {filename}\n")

    except requests.exceptions.HTTPError as e:
        codigo = e.response.status_code if e.response else "?"
        if codigo == 404:
            msg = f"location_id {loc_id} no encontrado en la API (restaurante eliminado o URL incorrecta)"
            df_work.loc[idx, "tripadvisor_status"] = "not_found_api"
        elif codigo == 401:
            msg = "API key inválida — verifica TRIPADVISOR_API_KEY en el .env"
            df_work.loc[idx, "tripadvisor_status"] = "auth_error"
        elif codigo == 429:
            msg = "Rate limit superado — espera un momento y vuelve a ejecutar"
            df_work.loc[idx, "tripadvisor_status"] = "rate_limit"
        else:
            msg = f"Error HTTP {codigo}: {str(e)[:150]}"
            df_work.loc[idx, "tripadvisor_status"] = "error"

        df_work.loc[idx, "tripadvisor_error"] = msg
        log_event(global_logs, id_val, nombre, "tripadvisor_api",
                  df_work.loc[idx, "tripadvisor_status"], msg)
        print(f"   ❌ Error HTTP {codigo}: {msg[:80]}\n")

    except requests.exceptions.Timeout:
        msg = "Timeout al llamar a la API"
        df_work.loc[idx, "tripadvisor_status"] = "timeout"
        df_work.loc[idx, "tripadvisor_error"]  = msg
        log_event(global_logs, id_val, nombre, "tripadvisor_api", "timeout", msg)
        print(f"   ⏱️  Timeout\n")

    except Exception as e:
        msg = str(e)[:300]
        df_work.loc[idx, "tripadvisor_status"] = "error"
        df_work.loc[idx, "tripadvisor_error"]  = msg
        log_event(global_logs, id_val, nombre, "tripadvisor_api", "error", msg)
        print(f"   ❌ Excepción: {msg[:80]}\n")

    finally:
        # Pausa corta entre llamadas (la API es estable, no necesita delays largos)
        if i < len(df_con_ta) - 1:
            random_delay(0.5, 1.2)

print("-" * 65)
print("✅ Consulta API completada.")
print(f"   data_extracted  : {(df_work['tripadvisor_status']=='data_extracted').sum()}")
print(f"   not_found_api   : {(df_work['tripadvisor_status']=='not_found_api').sum()}")
print(f"   rate_limit      : {(df_work['tripadvisor_status']=='rate_limit').sum()}")
print(f"   auth_error      : {(df_work['tripadvisor_status']=='auth_error').sum()}")
print(f"   error/timeout   : {df_work['tripadvisor_status'].isin(['error','timeout']).sum()}")
print(f"   JSON guardados en: {JSON_DIR}")


📡 Consultando TripAdvisor Content API
   Restaurantes a consultar       : 35
   Sin location_id (se omiten)    : 6
   API Key                        : ...06BC
-----------------------------------------------------------------

  ⚠️  Restaurantes SIN location_id (requieren revisión manual):
     - Bar De Tapas | URL: https://www.tripadvisor.co/Restaurant_Review-g294074-d128732...
     - Apolo Greek Street | URL: https://www.tripadvisor.com/Restaurants-g294074-c23-Bogota.h...
     - La trattoria del centro | URL: https://www.tripadvisor.co/Restaurant_Review-g294074-d140882...
     - LA TRATTORIA DEL BARRIO | URL: https://www.tripadvisor.co/Restaurant_Review-g294074-d150891...
     - Gyros Street | URL: https://www.tripadvisor.es/Restaurant_Review-g294074-d208466...
     - Chinita Bar de Tapas | URL: https://www.tripadvisor.co/Restaurant_Review-g294074-d128732...

  [ 1/35] Vapiano Colombia Restaurante Italiano
   location_id: 12657249
   ✅ rating=3.7 | reviews=190 | N.º 550 de 6.664 Sitio

## Chunk 12: Validación de resultados y gestión de casos especiales

Verifica los resultados de la API e identifica casos que requieren atención:
- `not_found_api` → el location_id no está en la API (URL incorrecta o restaurante eliminado)
- `rate_limit` → se puede reintentar en el mismo chunk
- Sin datos → rating o reviews nulos pese a status `data_extracted`

También permite **reintentar** casos fallidos sin volver a correr todo el proceso.


In [25]:
# ── Resumen de resultados ─────────────────────────────────────────────────────
print("🔍 Validación de resultados de la API\n")
print("-" * 65)

# Casos exitosos
ok = df_work[df_work["tripadvisor_status"] == "data_extracted"]
print(f"✅ Extraídos correctamente    : {len(ok)}")

# Casos con datos nulos pese a status OK
ok_sin_datos = ok[ok["tripadvisor_rating"].isna() | ok["tripadvisor_n_reviews"].isna()]
if len(ok_sin_datos) > 0:
    print(f"⚠️  Con status OK pero sin datos: {len(ok_sin_datos)}")
    for _, r in ok_sin_datos.iterrows():
        print(f"     - {r[NAME_COL]} | rating={r['tripadvisor_rating']} | reviews={r['tripadvisor_n_reviews']}")

# Casos no encontrados en la API
not_found = df_work[df_work["tripadvisor_status"] == "not_found_api"]
if len(not_found) > 0:
    print(f"\n❌ No encontrados en la API ({len(not_found)}):")
    print("   Posibles causas: URL incorrecta, restaurante eliminado de Tripadvisor")
    for _, r in not_found.iterrows():
        print(f"   - {r[NAME_COL]}")
        print(f"     location_id: {r['tripadvisor_location_id']}")
        print(f"     URL        : {r['tripadvisor_url']}")

# Casos con rate limit (reintentar)
rate = df_work[df_work["tripadvisor_status"] == "rate_limit"]
if len(rate) > 0:
    print(f"\n⏳ Rate limit ({len(rate)} casos) — Reintentando en 60 segundos...")
    time.sleep(60)
    for idx, row in rate.iterrows():
        loc_id = str(row["tripadvisor_location_id"])
        nombre = str(row[NAME_COL])
        try:
            data   = obtener_detalles_tripadvisor(loc_id, TRIPADVISOR_API_KEY)
            parsed = parsear_respuesta_api(data)
            df_work.loc[idx, "tripadvisor_rating"]    = parsed["rating"]
            df_work.loc[idx, "tripadvisor_n_reviews"] = parsed["n_reviews"]
            df_work.loc[idx, "tripadvisor_ranking"]   = parsed["ranking_str"]
            df_work.loc[idx, "tripadvisor_status"]    = "data_extracted"
            df_work.loc[idx, "tripadvisor_error"]     = None
            print(f"   ✅ Reintento OK: {nombre}")
        except Exception as e:
            print(f"   ❌ Reintento fallido: {nombre} — {e}")
        random_delay(1.0, 2.0)

print("\n" + "-" * 65)
print("📊 Resumen final por status:")
resumen = df_work.groupby("tripadvisor_status", dropna=False).size().reset_index(name="n")
print(resumen.to_string(index=False))


🔍 Validación de resultados de la API

-----------------------------------------------------------------
✅ Extraídos correctamente    : 35
⚠️  Con status OK pero sin datos: 2
     - Huerta Coctelería Artesanal | rating=None | reviews=0
     - Greko | rating=None | reviews=0

-----------------------------------------------------------------
📊 Resumen final por status:
tripadvisor_status  n
    data_extracted 35
   found_uncertain  6


## Chunk 14: Verificación del append final

In [26]:
print("🔍 Verificación del DataFrame final\n")

print("Columnas originales (deben estar intactas):")
for col in ORIGINAL_COLS:
    ok = col in df_work.columns
    print(f"   {'✅' if ok else '❌'} {col}")

print("\nColumnas nuevas añadidas:")
for col in NEW_COLS:
    ok = col in df_work.columns
    if ok:
        n_vals = df_work[col].notna().sum()
        print(f"   ✅ {col:<35} ({n_vals}/{len(df_work)} filas con datos)")
    else:
        print(f"   ❌ {col} — NO ENCONTRADA")

print(f"\nTotal filas    : {len(df_work)}")
print(f"Total columnas : {len(df_work.columns)}")

# Verificar que 'rating' y 'num_reviews' originales no fueron modificados
orig_rating_ok    = df_work["rating"].equals(df_raw["rating"])
orig_reviews_ok   = df_work["num_reviews"].equals(df_raw["num_reviews"])
print(f"\nColumnas originales clave intactas:")
print(f"   rating      : {'✅ Sin cambios' if orig_rating_ok else '❌ MODIFICADA'}")
print(f"   num_reviews : {'✅ Sin cambios' if orig_reviews_ok else '❌ MODIFICADA'}")


🔍 Verificación del DataFrame final

Columnas originales (deben estar intactas):
   ✅ id
   ✅ place_id
   ✅ nombre
   ✅ direccion
   ✅ lat
   ✅ lon
   ✅ rating
   ✅ num_reviews
   ✅ google_maps_url
   ✅ tipos
   ✅ query_origen
   ✅ barrio
   ✅ cocina
   ✅ no_visitar
   ✅ sample
   ✅ encuesta

Columnas nuevas añadidas:
   ✅ tripadvisor                         (41/41 filas con datos)
   ✅ tripadvisor_url                     (41/41 filas con datos)
   ✅ tripadvisor_location_id             (35/41 filas con datos)
   ✅ tripadvisor_rating                  (33/41 filas con datos)
   ✅ tripadvisor_n_reviews               (35/41 filas con datos)
   ✅ tripadvisor_ranking                 (33/41 filas con datos)
   ✅ tripadvisor_status                  (41/41 filas con datos)
   ✅ tripadvisor_error                   (0/41 filas con datos)
   ✅ tripadvisor_txt_file                (35/41 filas con datos)

Total filas    : 41
Total columnas : 25

Columnas originales clave intactas:
   rating      : ✅ 

## Chunk 15: Exportación final de la base enriquecida

In [31]:
# ── Ordenar columnas: originales primero, nuevas al final ─────────────────────
cols_orig = [c for c in ORIGINAL_COLS if c in df_work.columns]
cols_new  = [c for c in NEW_COLS      if c in df_work.columns]
cols_extra = [c for c in df_work.columns if c not in cols_orig + cols_new]

df_final = df_work[cols_orig + cols_new + cols_extra].copy()

# ── Exportar ──────────────────────────────────────────────────────────────────
output_path =  CLEAN_DIR/ "base_restaurantes_enriquecida_tripadvisor.xlsx"
df_final.to_excel(output_path, index=False)

print("✅ Base enriquecida exportada exitosamente.")
print(f"   Archivo  : {output_path}")
print(f"   Filas    : {len(df_final)}")
print(f"   Columnas : {len(df_final.columns)}")
print()
print("📊 Resumen final:")
print(f"   Con Tripadvisor            : {(df_final['tripadvisor']==1).sum()}")
print(f"   Sin Tripadvisor            : {(df_final['tripadvisor']==0).sum()}")
print(f"   Con rating extraído        : {df_final['tripadvisor_rating'].notna().sum()}")
print(f"   Con n_reviews extraído     : {df_final['tripadvisor_n_reviews'].notna().sum()}")
print(f"   Con ranking extraído       : {df_final['tripadvisor_ranking'].notna().sum()}")
print()

cols_preview = ["id","nombre","tripadvisor","tripadvisor_rating",
                "tripadvisor_n_reviews","tripadvisor_ranking","tripadvisor_status"]
display(df_final[cols_preview].head(15))


✅ Base enriquecida exportada exitosamente.
   Archivo  : C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect\Data\Clean\base_restaurantes_enriquecida_tripadvisor.xlsx
   Filas    : 41
   Columnas : 25

📊 Resumen final:
   Con Tripadvisor            : 41
   Sin Tripadvisor            : 0
   Con rating extraído        : 33
   Con n_reviews extraído     : 35
   Con ranking extraído       : 33



,id,nombre,tripadvisor,tripadvisor_rating,tripadvisor_n_reviews,tripadvisor_ranking,tripadvisor_status
0,1,Vapiano Colombia Restaurante Italiano,1,3.7,190,N.º 550 de 6.664 Sitios Para Comer en Bogotá,data_extracted
1,2,Storia D'Amore zona T,1,4.9,5428,N.º 6 de 6.664 Sitios Para Comer en Bogotá,data_extracted
2,5,Restaurante español Gaudí,1,3.9,231,N.º 393 de 6.664 Sitios Para Comer en Bogotá,data_extracted
3,9,La Fabbrica,1,4.3,825,N.º 143 de 6.664 Sitios Para Comer en Bogotá,data_extracted
4,15,Griego Rooftop | Crossover Rooftop & Nightclub...,1,4.5,2,65 de 288 Vida nocturna en Bogotá,data_extracted
5,16,Salonika,1,4.5,79,N.º 377 de 6.664 Sitios Para Comer en Bogotá,data_extracted
6,17,Primi,1,4.3,529,N.º 160 de 6.664 Sitios Para Comer en Bogotá,data_extracted
7,19,Santorini Rooftop,1,3.9,114,N.º 535 de 6.664 Sitios Para Comer en Bogotá,data_extracted
8,21,Huerta Coctelería Artesanal,1,None,0,None,data_extracted
9,22,La Tapería,1,4.3,419,N.º 247 de 6.664 Sitios Para Comer en Bogotá,data_extracted


## Chunk 16: Logs, control de errores y recomendaciones finales

In [28]:
# ── Exportar log de eventos ───────────────────────────────────────────────────
logs_path = LOGS_DIR / "tripadvisor_api_log.xlsx"
df_logs   = pd.DataFrame(global_logs)

if not df_logs.empty:
    df_logs.to_excel(logs_path, index=False)
    print(f"✅ Log exportado: {logs_path}")
    print(f"   Eventos registrados: {len(df_logs)}")
    print("\nResumen por etapa y status:")
    resumen_log = df_logs.groupby(["etapa","status"]).size().reset_index(name="n")
    print(resumen_log.to_string(index=False))
else:
    print("⚠️  No hay eventos en el log.")


✅ Log exportado: C:\Users\anton\OneDrive\Documentos\GitHub\DoingEconomics_Proyect\outputs\logs\tripadvisor_api_log.xlsx
   Eventos registrados: 156

Resumen por etapa y status:
          etapa         status   n
busqueda_serper             ok  41
tripadvisor_api data_extracted 115


In [29]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║         RECOMENDACIONES FINALES — v2 (API oficial)              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. CASOS not_found_api                                          ║
║     El location_id no existe en la API.                          ║
║     → Busca el restaurante manualmente en tripadvisor.co/es     ║
║     → Copia la URL, extrae el nuevo '-d{id}-'                   ║
║     → Actualiza 'tripadvisor_location_id' en df_work            ║
║     → Vuelve a correr solo el Chunk 11 para ese restaurante     ║
║                                                                  ║
║  2. CASOS found_no_id (sin location_id)                          ║
║     → Busca el restaurante en Tripadvisor                        ║
║     → Extrae el ID de la URL final (después de redirecciones)   ║
║     → Añádelo en 'tripadvisor_location_id' y re-corre Chunk 11  ║
║                                                                  ║
║  3. CONSULTA DE DATOS ADICIONALES DE LA API                      ║
║     El JSON guardado en outputs/json/ tiene muchos más campos:   ║
║     price_level, cuisine, hours, phone, website, etc.            ║
║     Puedes añadirlos a df_work con parsear_respuesta_api().     ║
║                                                                  ║
║  4. MONITOREO DE LLAMADAS API                                    ║
║     Revisa tu uso mensual en:                                    ║
║     https://tripadvisor-content-api.readme.io/reference/faq     ║
║     Con 41 restaurantes usas < 1% de las 5.000 calls gratuitas. ║
║                                                                  ║
║  5. COLUMNAS ORIGINALES PROTEGIDAS                               ║
║     'rating' y 'num_reviews' originales NO fueron modificadas.   ║
║     Los datos de Tripadvisor están en columnas con prefijo       ║
║     'tripadvisor_' para evitar cualquier ambigüedad.             ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")



╔══════════════════════════════════════════════════════════════════╗
║         RECOMENDACIONES FINALES — v2 (API oficial)              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. CASOS not_found_api                                          ║
║     El location_id no existe en la API.                          ║
║     → Busca el restaurante manualmente en tripadvisor.co/es     ║
║     → Copia la URL, extrae el nuevo '-d{id}-'                   ║
║     → Actualiza 'tripadvisor_location_id' en df_work            ║
║     → Vuelve a correr solo el Chunk 11 para ese restaurante     ║
║                                                                  ║
║  2. CASOS found_no_id (sin location_id)                          ║
║     → Busca el restaurante en Tripadvisor                        ║
║     → Extrae el ID de la URL final (después de redirecciones)   ║
║     → Añádelo en 'tripadvisor_locatio